# "Who Hunts Who?"
## Comparing Three Wikipedia-based Knowledge Graph Construction Methods for Predator–Prey Relations

**NLP MAI SS2026 – Group Project**  
**Members:** Zoltan Bek, Armin Lohse, Nino Reil, Andreas Wallnöfer

---

### Overview

This notebook implements and compares three approaches to constructing a Knowledge Graph (KG) of predator–prey relations using Wikipedia as the sole data source. All methods operate on the same corpus of ~20 animal pages and are evaluated against a manually curated gold standard.

| Section | Description |
|---|---|
| 0 | Setup & Imports |
| 1 | Dataset – Wikipedia Corpus |
| 2 | Gold Standard |
| 3 | Method A – Co-occurrence KG |
| 4 | Method B – Dependency-based KG |
| 5 | Method C – Wikilinks KG |
| 6 | Evaluation & Comparison |
| 7 | Discussion |


---
## Section 0 – Setup & Imports

In [1]:
# 0.1  Install dependencies (run once; comment out after first execution)
# After running this cell for the first time, restart the kernel and comment these lines out.
!pip install -q wikipedia-api spacy networkx pandas matplotlib seaborn
!python -m spacy download en_core_web_md -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [3]:
# 0.2  Imports

# --- standard library ---
import json
import time
import re
from pathlib import Path

# --- third-party ---
import wikipediaapi
import spacy
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Notebook display settings
pd.set_option("display.max_colwidth", 120)
%matplotlib inline
print("✅ All imports successful.")

✅ All imports successful.


---
## Section 1 – Dataset: Wikipedia Corpus

We work with a controlled corpus of **~20 Wikipedia animal pages** forming a terrestrial food web.

### 1.1 Animal Lists

Animals are grouped by trophic role to allow targeted evaluation later.

| Group | Animals |
|---|---|
| **Top predators** | Hawk, Eagle, Owl, Fox, Wolf, Snake |
| **Mid-level** (predator + prey) | Frog, Lizard, Spider, Bird, Shrew, Bat |
| **Lower trophic / common prey** | Mouse, Rabbit, Squirrel, Grasshopper, Cricket, Caterpillar, Fly, Beetle |

These form example chains such as **Hawk → Snake → Frog → Grasshopper**.

> **Implementation note:** We start with general Wikipedia pages (e.g. `"Hawk"`) and may switch to more specific species pages (e.g. `"Red-tailed hawk"`) if the general page lacks clear diet statements.

In [4]:
# 1.1  Animal lists and Wikipedia page title mapping

PREDATORS = ["Hawk", "Eagle", "Owl", "Fox", "Wolf", "Snake"]

MID_LEVEL = ["Frog", "Lizard", "Spider", "Bird", "Shrew", "Bat"]

PREY = ["Mouse", "Rabbit", "Squirrel", "Grasshopper", "Cricket", "Caterpillar", "Fly", "Beetle"]

ALL_ANIMALS = PREDATORS + MID_LEVEL + PREY

# Map animal label → exact Wikipedia page title.
# Most match directly; override here where the general page title differs.
WIKI_PAGE_MAP = {
    "Hawk":        "Hawk",
    "Eagle":       "Eagle",
    "Owl":         "Owl",
    "Fox":         "Fox",
    "Wolf":        "Wolf",
    "Snake":       "Snake",
    "Frog":        "Frog",
    "Lizard":      "Lizard",
    "Spider":      "Spider",
    "Bird":        "Bird",
    "Shrew":       "Shrew",
    "Bat":         "Bat",
    "Mouse":       "Mouse",
    "Rabbit":      "Rabbit",
    "Squirrel":    "Squirrel",
    "Grasshopper": "Grasshopper",
    "Cricket":     "Cricket (insect)",   # disambiguation: cricket sport vs insect
    "Caterpillar": "Caterpillar",
    "Fly":         "Fly",
    "Beetle":      "Beetle",
}

# Trophic colour map — used later for graph visualisations
TROPHIC_COLOR = {
    animal: color
    for animals, color in [
        (PREDATORS, "#e05c5c"),   # red   – top predators
        (MID_LEVEL, "#f0a500"),   # amber – mid-level
        (PREY,      "#4da06d"),   # green – lower trophic
    ]
    for animal in animals
}

print(f"Animal corpus: {len(ALL_ANIMALS)} animals total")
print(f"  Predators : {PREDATORS}")
print(f"  Mid-level : {MID_LEVEL}")
print(f"  Prey      : {PREY}")

Animal corpus: 20 animals total
  Predators : ['Hawk', 'Eagle', 'Owl', 'Fox', 'Wolf', 'Snake']
  Mid-level : ['Frog', 'Lizard', 'Spider', 'Bird', 'Shrew', 'Bat']
  Prey      : ['Mouse', 'Rabbit', 'Squirrel', 'Grasshopper', 'Cricket', 'Caterpillar', 'Fly', 'Beetle']


### 1.2 Fetching Wikipedia Pages

For each animal we retrieve:
- **Full plain text** of the page (for Methods A and B)
- **Internal wikilinks** (list of pages linked from this page, for Method C)

Pages are fetched once and cached locally (as a dict or JSON file) to avoid repeated API calls during development.

In [5]:
# 1.2  Fetch and cache Wikipedia pages
#
# Strategy:
#   - On first run, fetch all pages from the Wikipedia API and save to corpus_cache.json.
#   - On subsequent runs, load from cache to avoid repeated API calls.
#   - For each animal we store:
#       'text'  : full plain-text content of the page
#       'links' : list of Wikipedia page titles linked from this page (for Method C)
#       'url'   : canonical Wikipedia URL (for gold-standard citations)

CACHE_PATH = Path("corpus_cache.json")

def fetch_corpus(page_map: dict, force_refresh: bool = False) -> dict:
    """Fetch Wikipedia pages for all animals. Loads from cache if available."""

    if CACHE_PATH.exists() and not force_refresh:
        print(f"📂 Loading corpus from cache: {CACHE_PATH}")
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    print("🌐 Fetching pages from Wikipedia API (this may take ~30 seconds)...")
    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="NLP-MAI-SS2026/1.0 (university project; contact: student@example.com)"
    )

    corpus = {}
    for animal, title in page_map.items():
        page = wiki.page(title)
        if not page.exists():
            print(f"  ⚠️  Page not found for '{animal}' (title: '{title}') — skipping.")
            continue

        corpus[animal] = {
            "title": page.title,
            "url":   page.fullurl,
            "text":  page.text,
            "links": list(page.links.keys()),   # titles of all linked Wikipedia pages
        }
        print(f"  ✅ {animal:15s}  ({len(page.text):,} chars, {len(page.links)} links)")

    # Persist to disk
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Corpus cached to {CACHE_PATH}")
    return corpus


# Run fetch (set force_refresh=True to re-download everything)
wiki_corpus = fetch_corpus(WIKI_PAGE_MAP, force_refresh=False)

🌐 Fetching pages from Wikipedia API (this may take ~30 seconds)...
  ✅ Hawk             (10,341 chars, 127 links)
  ✅ Eagle            (21,402 chars, 316 links)
  ✅ Owl              (31,916 chars, 541 links)
  ✅ Fox              (16,959 chars, 581 links)
  ✅ Wolf             (56,540 chars, 1150 links)
  ✅ Snake            (60,712 chars, 680 links)
  ✅ Frog             (79,353 chars, 775 links)
  ✅ Lizard           (28,181 chars, 364 links)
  ✅ Spider           (55,392 chars, 623 links)
  ✅ Bird             (77,772 chars, 1440 links)
  ✅ Shrew            (7,518 chars, 703 links)
  ✅ Bat              (50,398 chars, 451 links)
  ✅ Mouse            (7,955 chars, 130 links)
  ✅ Rabbit           (40,958 chars, 745 links)
  ✅ Squirrel         (13,401 chars, 652 links)
  ✅ Grasshopper      (30,758 chars, 385 links)
  ✅ Cricket          (26,234 chars, 353 links)
  ✅ Caterpillar      (16,177 chars, 211 links)
  ✅ Fly              (35,324 chars, 721 links)
  ✅ Beetle           (64,770 chars, 957 

### 1.3 Corpus Inspection

Quick sanity check: print page lengths and a sample sentence from each page to confirm the data looks as expected.

In [6]:
# 1.3  Corpus inspection — page lengths and a sample sentence per animal

rows = []
for animal, data in wiki_corpus.items():
    # Grab the first sentence that contains the animal name as a quick sanity snippet
    sentences = re.split(r'(?<=[.!?])\s+', data["text"])
    snippet = next(
        (s.strip() for s in sentences if animal.lower() in s.lower()),
        sentences[0].strip() if sentences else "(no text)"
    )
    rows.append({
        "animal":     animal,
        "trophic":    "predator" if animal in PREDATORS else ("mid" if animal in MID_LEVEL else "prey"),
        "chars":      len(data["text"]),
        "n_links":    len(data["links"]),
        "url":        data["url"],
        "snippet":    snippet[:200] + ("..." if len(snippet) > 200 else ""),
    })

corpus_df = pd.DataFrame(rows).sort_values(["trophic", "animal"]).reset_index(drop=True)
display(corpus_df[["animal", "trophic", "chars", "n_links", "snippet"]])

,animal,trophic,chars,n_links,snippet
0,Bat,mid,50398,451,"Bats (order Chiroptera ) are winged mammals, the only mammals capable of true and sustained flight."
1,Bird,mid,77772,1440,"Birds are a group of warm-blooded vertebrate animals constituting the class Aves, characterised by feathers, toothle..."
2,Frog,mid,79353,775,"A frog is any member of a diverse and largely semiaquatic group of short-bodied, tailless amphibian vertebrates comp..."
3,Lizard,mid,28181,364,Lizard is the common name used for all squamate reptiles other than snakes (and to a lesser extent amphisbaenians or...
4,Shrew,mid,7518,703,Shrews (family Soricidae) are small mole-like mammals classified in the order Eulipotyphla.
5,Spider,mid,55392,623,"Spiders (order Araneae) are air-breathing arthropods that have eight limbs, chelicerae with fangs generally able to ..."
6,Eagle,predator,21402,316,Eagle is the common name for certain large birds of prey within the family of the Accipitridae.
7,Fox,predator,16959,581,Foxes are small-to-medium-sized omnivorous mammals belonging to several genera of the family Canidae.
8,Hawk,predator,10341,127,Hawks are birds of prey of the family Accipitridae.
9,Owl,predator,31916,541,"Owls are birds from the order Strigiformes (), which includes over 200 species of mostly solitary and nocturnal bird..."


### 1.4 Debug: Corpus Health Check

Verifies that every animal in `ALL_ANIMALS` was fetched successfully, that no page is suspiciously short, and that the link lists are populated. This cell is safe to re-run at any time.

In [7]:
# 1.4  Debug — corpus health check

MIN_TEXT_CHARS  = 500    # pages shorter than this are likely disambiguation pages
MIN_LINK_COUNT  = 5     # pages with very few links are suspicious

issues   = []
ok_count = 0

print("=" * 60)
print(f"  Corpus health check  ({len(ALL_ANIMALS)} animals expected)")
print("=" * 60)

for animal in ALL_ANIMALS:
    if animal not in wiki_corpus:
        msg = f"❌ MISSING   {animal:15s}  — page was not fetched at all"
        print(msg); issues.append(msg)
        continue

    data   = wiki_corpus[animal]
    chars  = len(data["text"])
    nlinks = len(data["links"])
    animal_issues = []

    if chars < MIN_TEXT_CHARS:
        animal_issues.append(f"text too short ({chars} chars)")
    if nlinks < MIN_LINK_COUNT:
        animal_issues.append(f"very few links ({nlinks})")

    if animal_issues:
        msg = f"⚠️  WARNING  {animal:15s}  — {'; '.join(animal_issues)}"
        print(msg); issues.append(msg)
    else:
        print(f"✅ OK        {animal:15s}  {chars:>8,} chars   {nlinks:>4} links")
        ok_count += 1

print("=" * 60)
print(f"\nResult: {ok_count}/{len(ALL_ANIMALS)} pages passed all checks.")
if issues:
    print(f"\n{len(issues)} issue(s) found — review warnings above before proceeding.")
else:
    print("\n🎉 All pages look good! Ready to proceed to Section 2.")

  Corpus health check  (20 animals expected)
✅ OK        Hawk               10,341 chars    127 links
✅ OK        Eagle              21,402 chars    316 links
✅ OK        Owl                31,916 chars    541 links
✅ OK        Fox                16,959 chars    581 links
✅ OK        Wolf               56,540 chars   1150 links
✅ OK        Snake              60,712 chars    680 links
✅ OK        Frog               79,353 chars    775 links
✅ OK        Lizard             28,181 chars    364 links
✅ OK        Spider             55,392 chars    623 links
✅ OK        Bird               77,772 chars   1440 links
✅ OK        Shrew               7,518 chars    703 links
✅ OK        Bat                50,398 chars    451 links
✅ OK        Mouse               7,955 chars    130 links
✅ OK        Rabbit             40,958 chars    745 links
✅ OK        Squirrel           13,401 chars    652 links
✅ OK        Grasshopper        30,758 chars    385 links
✅ OK        Cricket            26,234 chars

---
## Section 2 – Gold Standard

To evaluate Recall and F1 we need a set of **known-true** predator→prey edges.

### 2.1 Construction Methodology

Each gold-standard entry is manually extracted from an explicit Wikipedia sentence. An entry is included only if the sentence clearly states that animal A eats/hunts/preys on animal B — ambiguous or indirect statements are excluded.

Each entry records:
- `predator` – animal name (must be in `ALL_ANIMALS`)
- `prey` – animal name (must be in `ALL_ANIMALS`)
- `source_page` – Wikipedia page URL where the sentence was found
- `evidence` – the supporting sentence verbatim

> **Scope:** Evaluation only considers edges where **both** nodes are in our animal list, so we do not penalise methods for missing relations that involve out-of-scope entities.

In [ ]:
# 2.1  Gold standard as a list of dicts / DataFrame
# gold_standard = [
#   {"predator": "Hawk",  "prey": "Mouse",       "source_page": "...", "evidence": "..."},
#   {"predator": "Snake", "prey": "Frog",        "source_page": "...", "evidence": "..."},
#   ...
# ]
# gold_df = pd.DataFrame(gold_standard)
# gold_edges = set(zip(gold_df["predator"], gold_df["prey"]))
pass

### 2.2 Gold Standard Summary

Display the full gold standard table and a small network visualisation of the true food web.

In [ ]:
# 2.2  Display gold_df and draw the gold-standard graph
pass

---
## Section 3 – Method A: Co-occurrence KG (Baseline)

**Idea:** If a predator name and a prey name appear in the **same sentence** (or paragraph), add a directed edge predator→prey.

**Expected characteristics:**
- High recall — many true relations will be captured
- Lower precision — co-occurrence does not imply predation; noisy edges expected
- Fast to compute (no parsing required)

### 3.1 Algorithm

```
For each Wikipedia page P of animal A:
    Split text into sentences
    For each sentence S:
        For each animal B ≠ A in ALL_ANIMALS:
            If B mentioned in S:
                Add directed edge A → B   (A is the page animal; A is assumed predator)
                Add directed edge B → A   (B could equally be predator)
```

> **Direction heuristic:** Since co-occurrence is symmetric, we add edges in both directions and rely on filtering (source ∈ PREDATORS, target ∈ PREY) during evaluation.

> **Variants to explore:** sentence-level vs. paragraph-level window; case-insensitive matching; plural/singular normalisation.

In [ ]:
# 3.1  Co-occurrence extraction
# import time
# t0 = time.time()
# ...
# time_A = time.time() - t0
# edges_A = set(...)  # set of (predator, prey) tuples
# graph_A = nx.DiGraph(); graph_A.add_edges_from(edges_A)
pass

### 3.2 Method A – Results

Inspect the extracted edges and visualise the resulting graph.

In [ ]:
# 3.2  Print edge count, sample edges, draw graph_A
pass

---
## Section 4 – Method B: Dependency-based Relation KG

**Idea:** Use spaCy dependency parsing to find explicit predation verbs (e.g. *eats*, *hunts*, *preys on*, *feeds on*) and extract the syntactic subject (predator) and object (prey) of those verbs.

**Expected characteristics:**
- Higher precision — edges are grounded in explicit linguistic relations
- Lower recall — misses implicit or paraphrased diet statements
- Slower than Method A (requires full dependency parse)

### 4.1 Predation Verb & Pattern List

We define a set of predation-indicating verbs and patterns:

| Pattern type | Examples |
|---|---|
| Active verb | *eats, hunts, catches, kills, consumes, preys on, feeds on, devours* |
| Passive form | *is eaten by, is hunted by, is preyed upon by* |
| Nominal | *predator of, diet includes, diet consists of* |

In [ ]:
# 4.1  Define predation verb list and load spaCy model
# PREDATION_VERBS = {"eat", "hunt", "prey", "feed", "catch", "kill", "consume", "devour"}
# nlp = spacy.load("en_core_web_md")
pass

### 4.2 Algorithm

```
For each Wikipedia page P of animal A:
    Parse text with spaCy (dependency parse)
    For each sentence S:
        For each token T in S:
            If lemma(T) in PREDATION_VERBS:
                Extract syntactic subject(s) → candidate predator
                Extract syntactic object(s)  → candidate prey
                If both subject and object can be resolved to animals in ALL_ANIMALS:
                    Add edge subject → object
        Also handle passive constructions (agent = predator, patient = prey)
```

In [ ]:
# 4.2  Dependency-based extraction
# t0 = time.time()
# ...
# time_B = time.time() - t0
# edges_B = set(...)
# graph_B = nx.DiGraph(); graph_B.add_edges_from(edges_B)
pass

### 4.3 Method B – Results

Inspect extracted edges, highlight any interesting dependency parses, and visualise the graph.

In [ ]:
# 4.3  Print edge count, sample edges with evidence sentences, draw graph_B
pass

---
## Section 5 – Method C: Wikilinks KG

**Idea:** Use Wikipedia's internal link structure. If the page for animal A contains a hyperlink to animal B's page, add a directed edge A→B.

**Expected characteristics:**
- Fast and simple — no NLP processing needed
- Captures general *association*, not necessarily direct predation
- Will include many false positives (links for reasons unrelated to diet)
- Useful as a structural baseline

### 5.1 Algorithm

```
For each Wikipedia page P of animal A:
    Retrieve the set of internal links L(P)
    For each animal B ≠ A in ALL_ANIMALS:
        If Wikipedia_title(B) ∈ L(P):
            Add directed edge A → B
```

> **Note:** Links are symmetric in practice; direction here reflects "A's page mentions B", which could mean A eats B or B eats A or simply that they share a habitat.

In [ ]:
# 5.1  Wikilinks extraction
# t0 = time.time()
# ...
# time_C = time.time() - t0
# edges_C = set(...)
# graph_C = nx.DiGraph(); graph_C.add_edges_from(edges_C)
pass

### 5.2 Method C – Results

Inspect links, identify cases where a link is and isn't a predation relation, and visualise the graph.

In [ ]:
# 5.2  Print edge count, sample edges, draw graph_C
pass

---
## Section 6 – Evaluation & Comparison

All three methods are evaluated against the gold standard using the same evaluation function. Only edges where the **source is in PREDATORS ∪ MID_LEVEL** and the **target is in MID_LEVEL ∪ PREY** are considered.

### 6.1 Evaluation Function

$$\text{Precision} = \frac{|\text{predicted} \cap \text{gold}|}{|\text{predicted}|}$$

$$\text{Recall} = \frac{|\text{predicted} \cap \text{gold}|}{|\text{gold}|}$$

$$\text{F1} = \frac{2 \cdot P \cdot R}{P + R}$$

In [ ]:
# 6.1  Evaluation helper function
# def evaluate(predicted_edges, gold_edges, label=""):
#     """Return precision, recall, F1 for a set of predicted edges vs. gold standard."""
#     ...
#     return {"method": label, "precision": P, "recall": R, "f1": F1,
#             "n_predicted": ..., "n_correct": ..., "runtime_s": ...}
pass

### 6.2 Run Evaluation for All Methods

In [ ]:
# 6.2  Compute metrics for A, B, C and collect into a results DataFrame
# results = pd.DataFrame([eval_A, eval_B, eval_C])
# display(results)
pass

### 6.3 Graph Structure Statistics

For each method's graph we report:
- Number of nodes and edges
- Graph density
- Average in-degree and out-degree
- Longest predation chain (longest directed path)

In [ ]:
# 6.3  Graph statistics for graph_A, graph_B, graph_C
pass

### 6.4 Comparison Plots

We produce the following visualisations:

1. **Grouped bar chart** – Precision / Recall / F1 per method
2. **Runtime bar chart** – seconds per method
3. **Edge Venn / upset plot** – overlap of predicted edges between methods
4. **Graph visualisations** – side-by-side NetworkX plots of the three graphs, coloured by trophic level

In [ ]:
# 6.4a  Precision / Recall / F1 grouped bar chart
pass

In [ ]:
# 6.4b  Runtime comparison bar chart
pass

In [ ]:
# 6.4c  Edge overlap analysis (which edges are shared / unique to each method)
pass

In [ ]:
# 6.4d  Side-by-side graph visualisations (trophic-level colour coding)
pass

### 6.5 Error Analysis

For each method, we inspect:
- **False positives** (edges predicted but not in gold) — what kinds of co-occurrences / links / parses cause them?
- **False negatives** (gold edges missed) — what linguistic patterns were not captured?

In [ ]:
# 6.5  False positive / false negative analysis per method
pass

---
## Section 7 – Discussion

### 7.1 Summary of Results

> *(To be filled in after experiments are complete.)*

Summarise the key quantitative findings: which method achieves the best F1, which has the highest precision, which has the highest recall, and how do they trade off against runtime.

### 7.2 Qualitative Observations

> *(To be filled in after experiments.)*

- What types of sentences does Method B struggle to parse correctly?
- Does Method C's link structure correlate well with actual predation?
- Are there interesting multi-hop chains recovered by any method?

### 7.3 Limitations

- Small corpus (~20 pages) limits generalisability.
- Gold standard is limited to our animal list — recall is measured within a closed world.
- Wikipedia writing style varies; some pages have clear diet sections, others do not.
- General animal pages (e.g. *"Snake"*) aggregate over many species and may be less precise than species pages.

### 7.4 Potential Extensions

- Scale to larger animal corpora (e.g., all mammals on Wikipedia).
- Add Method D: LLM-based relation extraction as a fourth comparison point.
- Use Wikidata structured data as an alternative gold standard.
- Extend to other relation types beyond predation (e.g., *symbiosis*, *competition*).

---
*End of notebook — NLP MAI SS2026*